# 0.18 — Theme discovery by novelty + bridging, tracked over time

**Goal (the whole pipeline in one line):** each week, *discover* candidate word-groups, **discard most**,
keep the few with a theme signature at **high recall**, **track** them across weeks, **promote** the ones
whose reach *grows*, and **LLM-confirm** the survivors — fully unsupervised (no theme keywords at detection).

This supersedes the earlier unseen-edge + Louvain version of 0.18. The lesson from that path: pair-density
bursts **fragment** a theme and surface its loudest single-company facet (Microsoft/OpenAI), which a fair
unsupervised LLM then rejects as a single-entity event. So we re-center the detection object on the **novel
anchor term** and its **bridging** across contexts, and we make persistence a *forward* confirmation, not a
birth-gate.

**Validation benchmark:** generative AI. ChatGPT 2022-11-30 · CHAT ETF 2023-05-17. Target: catch the genAI
anchor in **early December** and confirm it **months** before the ETF — without genAI keywords.

## The four operational tests (computable each week, no keywords)

1. **Novelty** — the anchor term is **baseline-unseen** (never appeared ≤ 2022-09-30). `chatgpt`, `openai`
   qualify; `microsoft`, `bankruptcy`, `opec` do not. *This is the strongest discard: most groups die here.*
2. **Bridging** — the anchor reaches **many distinct partners** in the week (`anchor_degree`). A theme is a
   hub fanning across contexts; an M&A dyad / single event is a tight clique with few partners.
3. **High-recall catch** — keep any novel anchor with `mentions ≥ MIN_MENTIONS` and `degree ≥ DEGREE_MIN`.
   **False positives are fine here** — precision comes from the next two stages.
4. **Forward persistence + growth** — *track* each caught anchor across weeks; **promote** when it has been
   alive ≥ `PERSIST_WEEKS` weeks **and** its reach hits a new high. One-off events die; spreading themes promote.

Then a final **LLM reject-confirm** on the assembled multi-actor group (generic, entity-free taxonomy).

In [ ]:
import os, re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"

ENV_PATH = _ROOT / ".env"                      # OpenAI creds (same stack as 0.5/0.6)
if ENV_PATH.exists():
    for _l in ENV_PATH.read_text().splitlines():
        _l = _l.strip()
        if "=" in _l and not _l.startswith("#"):
            _k, _, _v = _l.partition("=")
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

BASELINE_END = pd.Timestamp("2022-09-30")
DISCOVERY_START = pd.Timestamp("2022-10-01")
DISCOVERY_END = pd.Timestamp("2023-02-28")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")

FREQ = "W-MON"; REP_HL = 6
MIN_MENTIONS = 3        # candidate anchor must appear >= this many times in the week
DEGREE_MIN = 10         # catch: novel anchor must reach >= this many distinct partners
PERSIST_WEEKS = 2       # promote: alive >= this many weeks AND reach at a new high
LLM_MAX_GROUPS = 25     # cap LLM-confirm calls (top promoted groups by reach)

TERM_STOP = set(ENGLISH_STOP_WORDS) | {
    "says", "said", "new", "year", "week", "report", "shares", "stock",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
}
HINT = re.compile(r"chatgpt|openai|chatbot|generative|\bgpt-", re.I)   # validation only

def normalize_terms(v):
    if isinstance(v, np.ndarray): return v.tolist()
    return list(v) if isinstance(v, (list, tuple)) else []
def filter_terms(ts): return [t for t in ts if len(t) >= 3 and not any(tok in TERM_STOP for tok in t.split())]
def ek(a, b): return (a, b) if a < b else (b, a)
def week_ts(w): return pd.Period(w, freq=FREQ).start_time
def entropy(counts):
    a = np.array([c for c in counts if c > 0], float)
    return float(-(a / a.sum() * np.log(a / a.sum())).sum()) if a.sum() else 0.0
print("setup ok ·", f"{DISCOVERY_START.date()} -> {DISCOVERY_END.date()} · MIN_MENTIONS={MIN_MENTIONS} DEGREE_MIN={DEGREE_MIN}")

In [ ]:
def detect(news, baseline_end, ds, de):
    """Novelty census: one row per (baseline-unseen anchor term, week) with bridging degree."""
    df = news.copy()
    df["terms_f"] = df["terms"].map(normalize_terms).map(filter_terms)
    df["week"] = df["date"].dt.to_period(FREQ).astype(str)

    baseline_terms = set()                                  # §1 novelty reference
    for tl in df.loc[df.date <= baseline_end, "terms_f"]:
        baseline_terms.update(tl)
    weeks = sorted(df["week"].unique(), key=week_ts)
    disc = [w for w in weeks if ds <= week_ts(w) <= de]

    rows = []
    for week in disc:
        grp = df.loc[df.week == week]
        twc, adj = Counter(), defaultdict(Counter)
        for tl in grp["terms_f"]:
            s = set(tl)
            for t in s:
                twc[t] += 1
            for a, b in combinations(sorted(s), 2):
                adj[a][b] += 1; adj[b][a] += 1
        for t, cnt in twc.items():
            if t in baseline_terms or cnt < MIN_MENTIONS:    # novel + min support (high recall)
                continue
            partners = adj[t]                                # §2 bridging
            reps = [hl for hl, tl in zip(grp["Headline"], grp["terms_f"]) if t in set(tl)][:REP_HL]
            rows.append({
                "week": week, "anchor": t, "mentions": int(cnt),
                "anchor_degree": len(partners), "partner_entropy": round(entropy(list(partners.values())), 3),
                "top_partners": [p for p, _ in partners.most_common(12)], "rep_headlines": reps,
            })
    return pd.DataFrame(rows)

In [ ]:
# --- load (same inputs as before) and run the novelty census ---
meta = pd.read_parquet(OUTPUT_DIR / "genai_full_meta.parquet")
terms = pd.read_parquet(OUTPUT_DIR / "genai_graph_terms.parquet")
meta["date"] = pd.to_datetime(meta["date"]).dt.normalize()
terms["date"] = pd.to_datetime(terms["date"]).dt.normalize()
terms = terms.drop_duplicates(["Headline", "date"], keep="first")
news = meta.merge(terms[["Headline", "date", "terms"]], on=["Headline", "date"], how="left")
news = news.loc[news.date <= DISCOVERY_END]

R = detect(news, BASELINE_END, DISCOVERY_START, DISCOVERY_END)
R["genai"] = R["anchor"].str.contains(HINT, na=False)
print(f"{len(R):,} (novel anchor x week) rows · {R.anchor.nunique():,} distinct novel anchors")
print("\ngenAI anchors — weekly mentions / bridging degree:")
print(R.loc[R.genai, ["week", "anchor", "mentions", "anchor_degree"]]
      .sort_values(["week", "anchor"]).to_string(index=False))

In [ ]:
# ---------- §3 high-recall CATCH (false positives OK) ----------
R["caught"] = (R.mentions >= MIN_MENTIONS) & (R.anchor_degree >= DEGREE_MIN)
per_week = R[R.caught].groupby("week").anchor.nunique()
print(f"caught (anchor x week): {int(R.caught.sum()):,} · distinct caught anchors: {R[R.caught].anchor.nunique():,}")
print(f"caught novel anchors per week: median {int(per_week.median())}, max {int(per_week.max())}")
print("\ngenAI anchors caught?  ",
      R[R.genai & R.caught][["week", "anchor", "anchor_degree"]].to_string(index=False))

In [ ]:
# ---------- §4 cross-week TRACKING + promote on forward growth ----------
def promote(R):
    out = []
    for a, sub in R[R.caught].groupby("anchor"):
        sub = sub.sort_values("week", key=lambda s: s.map(week_ts))
        wk, deg = list(sub.week), list(sub.anchor_degree)
        pw = None
        for i in range(len(wk)):
            if (i + 1) >= PERSIST_WEEKS and deg[i] >= max(deg[:i] or [0]):   # alive + reach new high
                pw = wk[i]; break
        out.append({"anchor": a, "first_caught": wk[0], "n_weeks": len(wk),
                    "deg_first": deg[0], "deg_max": max(deg), "promoted_week": pw,
                    "genai": bool(HINT.search(a))})
    return pd.DataFrame(out)

P = promote(R)
promoted = P[P.promoted_week.notna()].copy()
print(f"promoted anchors: {len(promoted)} of {len(P)} caught  "
      f"(promotion = alive>={PERSIST_WEEKS} wks AND reach at new high)")
print("\ngenAI anchors — lifecycle:")
print(P[P.genai].sort_values("first_caught")[
      ["anchor", "first_caught", "n_weeks", "deg_first", "deg_max", "promoted_week"]].to_string(index=False))
print("\ntop promoted anchors by reach (FP inspection):")
print(promoted.sort_values("deg_max", ascending=False).head(25)[
      ["anchor", "first_caught", "promoted_week", "deg_max", "genai"]].to_string(index=False))

In [ ]:
# ---------- assemble promoted anchors into themes + unsupervised LLM confirm ----------
from typing import Literal
from pydantic import BaseModel

pset = set(promoted.anchor)
G = nx.Graph(); G.add_nodes_from(pset)
for _, r in R[R.anchor.isin(pset) & R.caught].iterrows():        # link anchors that co-occur
    for p in r.top_partners:
        if p in pset:
            G.add_edge(r.anchor, p)
groups = [sorted(c) for c in nx.connected_components(G)]
def grp_headlines(anchors):
    hl = []
    for h in R[R.anchor.isin(anchors) & R.caught].sort_values("week", key=lambda s: s.map(week_ts)).rep_headlines:
        hl += list(h)
    seen, out = set(), []
    for h in hl:
        k = h.lower()
        if k not in seen:
            seen.add(k); out.append(h)
    return out[:8]
group_df = pd.DataFrame({"anchors": groups})
group_df["reach"] = group_df.anchors.map(lambda a: int(P.set_index("anchor").loc[a, "deg_max"].max()))
group_df["genai"] = group_df.anchors.map(lambda a: any(HINT.search(x) for x in a))
group_df = group_df.sort_values("reach", ascending=False).reset_index(drop=True)

REJECT_SYS = (
    "You are a conservative FILTER that REMOVES clusters of news headlines that are NOT emerging themes. "
    "You never decide what IS a theme; you only flag clusters that CLEARLY fall into one of three non-theme "
    "categories, judging ONLY from the headlines shown. If a cluster does not clearly match a reject "
    "category, you MUST keep it. When in doubt, KEEP.\n\n"
    "1. single_entity_event - ONE company/person's idiosyncratic event (bankruptcy, fraud, lawsuit, a single "
    "firm's earnings or M&A, a named individual) with no broader multi-actor narrative.\n"
    "2. macro_market_aggregate - generic market/macro conditions: rates, inflation, FX, bond yields, equity "
    "indices, central-bank policy, broad commodity moves, GDP.\n"
    "3. boilerplate_wire - wire formatting, calendars, generic 'shares rise/fall', routine corporate PR.\n\n"
    "KEEP anything describing a SPECIFIC technological, industrial, product, or policy development connecting "
    "multiple actors - even if reported via a deal or an event."
)
class Reject(BaseModel):
    verdict: Literal["keep", "reject"]
    category: Literal["single_entity_event", "macro_market_aggregate", "boilerplate_wire", "none"]
    reason: str
_client = None
def judge(anchors, headlines):
    global _client
    from openai import OpenAI
    if _client is None:
        _client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE") or None)
    user = ("Cluster terms: " + ", ".join(anchors) + "\nRepresentative headlines:\n"
            + "\n".join(f"- {h}" for h in headlines) + "\n\nClassify this cluster.")
    r = _client.beta.chat.completions.parse(
        model=os.environ.get("OPENAI_DEFAULT_MODEL", "gpt-4o-mini"), temperature=0, response_format=Reject,
        messages=[{"role": "system", "content": REJECT_SYS}, {"role": "user", "content": user}])
    p = r.choices[0].message.parsed
    return (p.verdict == "keep"), p.category, p.reason

verdicts = []
for _, g in group_df.head(LLM_MAX_GROUPS).iterrows():
    hl = grp_headlines(g.anchors)
    keep, cat, reason = judge(g.anchors, hl)
    verdicts.append({"keep": keep, "cat": cat, "reason": reason})
vdf = pd.DataFrame(verdicts)
group_df.loc[group_df.index[:len(vdf)], ["llm_keep", "llm_cat"]] = vdf[["keep", "cat"]].values
print(f"assembled {len(groups)} promoted themes · LLM-confirmed top {len(vdf)} "
      f"(kept {int(vdf.keep.sum())}, rejected {int((~vdf.keep).sum())})")

In [ ]:
# ---------- RESULT vs benchmark ----------
gen_caught = R[R.genai & R.caught]
gen_first = gen_caught.week.min() if len(gen_caught) else None
gen_prom = P[P.genai & P.promoted_week.notna()]
gen_prom_week = gen_prom.promoted_week.min() if len(gen_prom) else None
gai_group = group_df[group_df.genai]

print("=" * 70)
print("genAI benchmark (no keywords used at detection):")
print(f"  first CAUGHT week     : {gen_first}")
print(f"  PROMOTED (confirmed)  : {gen_prom_week}")
print(f"  ChatGPT launch        : {CHATGPT_LAUNCH.date()}")
print(f"  CHAT ETF inception    : {INCEPTION.date()}")
if gen_prom_week:
    lead = (INCEPTION - week_ts(gen_prom_week)).days
    print(f"  lead vs CHAT ETF      : {lead} days (~{lead//30} months) before inception")
if len(gai_group):
    g0 = gai_group.iloc[0]
    print(f"  genAI theme group     : {', '.join(g0.anchors[:10])}")
    print(f"  LLM verdict on group  : {'KEEP' if g0.get('llm_keep') else 'reject/' + str(g0.get('llm_cat'))}")
print("=" * 70)
print(f"\nfunnel: {R.anchor.nunique():,} novel anchors -> {R[R.caught].anchor.nunique():,} caught "
      f"-> {len(promoted)} promoted -> {len(groups)} themes -> {int(group_df.llm_keep.fillna(False).sum())} LLM-kept")
print("\nLLM-kept promoted themes (the shortlist a human would inspect):")
keptg = group_df[group_df.llm_keep == True].head(20)
for _, g in keptg.iterrows():
    star = " ★genAI" if g.genai else ""
    print(f"  reach {int(g.reach):4}  {', '.join(g.anchors[:8])}{star}")

R.to_parquet(OUTPUT_DIR / "genai_novelty_anchor_weeks.parquet", index=False)
P.to_parquet(OUTPUT_DIR / "genai_novelty_promotion.parquet", index=False)
print("\nsaved -> genai_novelty_anchor_weeks.parquet · genai_novelty_promotion.parquet")